In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

TRAINER_DIR = Path.cwd().resolve().parent 

RAW_DIR = TRAINER_DIR / "data" / "raw"

# You can adjust this depending on if you want plots inside 'notebooks/plots' or 'trainer.../plots'
PLOT_DIR = Path.cwd() / "plots" 
PLOT_DIR.mkdir(exist_ok=True)

DIVERSE_CSV = RAW_DIR / "angriness_dataset_diverse.csv"

if not DIVERSE_CSV.exists():
    raise FileNotFoundError(f"CSV file not found at {DIVERSE_CSV}")
    
CSV_PATH = DIVERSE_CSV

IDENTIFIER_COLS = ["game_id", "username", "created_at", "last_move_at"]

FEATURE_COLS = [
    "sleep_duration",
    "consecutive_losses",
    "awaken_duration",
    "avg_tpm",
    "move_cnt",
    "inaccuracy_cnt_player",
    "mistake_cnt_player",
    "blunder_cnt_player",
    "elo",
    "elo_diff",
    "player_color",
    "time_control_initial",
    "time_control_increment",
    "opponent_elo",
    "elo_gap",
    "move_cnt_player",
    "avg_tpm_seconds_player",
    "consecutive_losses_pregame",
    "acpl_player",
    "accuracy_player",
    "avg_ppm",
    "avg_celsius",
    "water_intake_ml",
    "avg_lux",
]

In [34]:
print(f"Loading: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
print(f"Unique players: {df['username'].nunique()}")

# --- Basic stats ---
print("\n=== Describe (numeric) ===")
print(df.describe().T.to_string())

Loading: C:\swe_repos\my-portfolio-and-vps-repos\Chess-Assistance\machine_learning\trainers\trainer-angriness-predictor\data\angriness_dataset_diverse.csv
Shape: (25249, 28)
Unique players: 12301

=== Describe (numeric) ===
                              count          mean           std           min           25%           50%           75%           max
sleep_duration              25249.0  7.003305e+00  1.499409e+00  3.000000e+00  5.980000e+00  6.990000e+00  8.020000e+00  1.200000e+01
consecutive_losses          25249.0  3.105866e-01  7.712551e-01  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  2.000000e+01
awaken_duration             25249.0  4.037130e+00  1.929954e+00  5.000000e-01  2.670000e+00  4.000000e+00  5.350000e+00  1.317000e+01
avg_tpm                     25249.0  8.156904e+00  6.208005e+00  7.614213e-02  3.689000e+00  7.336000e+00  1.136170e+01  9.940781e+01
move_cnt                    25249.0  6.796131e+01  2.799001e+01  2.000000e+00  4.600000e+01  6.600000e+01 

In [35]:
df.head()
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) > 0:
    print(f"\n=== Null counts ===\n{nulls.to_string()}")
else:
    print("\nNo nulls found.")


No nulls found.


In [36]:
# --- Duplicates ---
dupes = df.duplicated(subset=["game_id", "username"]).sum()
print(f"\nDuplicate (game_id, username) rows: {dupes}")


Duplicate (game_id, username) rows: 0


In [37]:
# --- ELO distribution ---
if "elo" in df.columns:
    bins = [0, 1200, 1600, 2000, 2400, 5000]
    labels = ["800-1200", "1200-1600", "1600-2000", "2000-2400", "2400+"]
    df["_bracket"] = pd.cut(df["elo"], bins=bins, labels=labels)
    print(f"\n=== Rows per ELO bracket ===\n{df['_bracket'].value_counts().sort_index().to_string()}")
    df.drop(columns=["_bracket"], inplace=True)


=== Rows per ELO bracket ===
_bracket
800-1200     1287
1200-1600    6000
1600-2000    6000
2000-2400    5962
2400+        6000


In [38]:
# --- Feature distributions ---
numeric_features = [c for c in FEATURE_COLS if c in df.columns and c != "player_color"]
available = df[numeric_features].select_dtypes(include=[np.number]).columns.tolist()

ncols = 4
nrows = (len(available) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = axes.flatten()

for i, col in enumerate(available):
    df[col].dropna().hist(bins=50, ax=axes[i], edgecolor="white", alpha=0.8)
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=7)

for j in range(len(available), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature Distributions", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(PLOT_DIR / "distributions.png", dpi=150)
print(f"\nSaved: {PLOT_DIR / 'distributions.png'}")
plt.close(fig)


Saved: c:\swe_repos\my-portfolio-and-vps-repos\Chess-Assistance\machine_learning\trainers\trainer-angriness-predictor\notebooks\plots\distributions.png


In [39]:
corr = df[available].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
    annot_kws={"size": 6}, linewidths=0.5)
ax.set_title("Feature Correlation Matrix")
fig.tight_layout()
fig.savefig(PLOT_DIR / "correlation_matrix.png", dpi=150)
print(f"Saved: {PLOT_DIR / 'correlation_matrix.png'}")
plt.close(fig)

Saved: c:\swe_repos\my-portfolio-and-vps-repos\Chess-Assistance\machine_learning\trainers\trainer-angriness-predictor\notebooks\plots\correlation_matrix.png


In [ ]:
# --- Highly correlated pairs ---
print("\n=== Highly correlated pairs (|r| > 0.8) ===")
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = [(col, row, upper.loc[row, col])
         for col in upper.columns for row in upper.index
         if abs(upper.loc[row, col]) > 0.8]
if pairs:
    for c1, c2, r in sorted(pairs, key=lambda x: -abs(x[2])):
        print(f"  {c1:30s} vs {c2:30s}  r={r:.3f}")
else:
    print("  None found.")

In [ ]:
# --- IoT feature stats ---
iot_cols = [c for c in ["avg_ppm", "avg_celsius", "water_intake_ml", "avg_lux"] if c in df.columns]
if iot_cols:
    print(f"\n=== IoT feature stats ===\n{df[iot_cols].describe().T.to_string()}")

# Preprocessing
Code from `2_preprocess.py`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

DROP_COLS = ["game_id", "username", "created_at", "last_move_at"]
REDUNDANT_COLS = ["consecutive_losses", "avg_tpm"]

TILT_BEHAVIORAL = [
    "consecutive_losses_pregame",
    "avg_tpm_seconds_player",
    "blunder_cnt_player",
    "mistake_cnt_player",
    "inaccuracy_cnt_player",
    "acpl_player",
    "accuracy_player",
]

TILT_CONTEXT = [
    "elo",
    "elo_diff",
    "opponent_elo",
    "elo_gap",
    "time_control_initial",
    "time_control_increment",
    "move_cnt",
    "move_cnt_player",
]

TILT_PHYSIOLOGICAL = [
    "sleep_duration",
    "awaken_duration",
    "avg_ppm",
    "avg_celsius",
    "water_intake_ml",
    "avg_lux",
]

CATEGORICAL_COLS = ["player_color"]

print(f"Loading: {CSV_PATH}")
df_prep = pd.read_csv(CSV_PATH)
print(f"Raw shape: {df_prep.shape}")

# --- Drop identifiers and redundant columns ---
to_drop = [c for c in DROP_COLS + REDUNDANT_COLS if c in df_prep.columns]
df_prep.drop(columns=to_drop, inplace=True)
print(f"After dropping identifiers/redundant: {df_prep.shape}")

# --- Encode player_color (white=0, black=1) ---
if "player_color" in df_prep.columns:
    df_prep["is_black"] = (df_prep["player_color"] == "black").astype(int)
    df_prep.drop(columns=["player_color"], inplace=True)

# --- Handle nulls ---
null_counts = df_prep.isnull().sum()
null_cols = null_counts[null_counts > 0]
if len(null_cols) > 0:
    print(f"\nNull columns before fill:\n{null_cols.to_string()}")
    for col in null_cols.index:
        if df_prep[col].dtype in [np.float64, np.int64, float, int]:
            df_prep[col].fillna(df_prep[col].median(), inplace=True)
        else:
            df_prep[col].fillna(0, inplace=True)

print(f"Nulls after fill: {df_prep.isnull().sum().sum()}")

# --- Keep unscaled copy in memory for later inspection ---
df_unscaled = df_prep.copy()

# --- Scale numeric features ---
all_features = TILT_BEHAVIORAL + TILT_CONTEXT + TILT_PHYSIOLOGICAL + ["is_black"]
available_prep = [c for c in all_features if c in df_prep.columns]
missing = [c for c in all_features if c not in df_prep.columns]
if missing:
    print(f"Warning: missing features: {missing}")

df_features = df_prep[available_prep].copy()

numeric_cols_prep = df_features.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
df_features[numeric_cols_prep] = scaler.fit_transform(df_features[numeric_cols_prep])

# --- Summary ---
print(f"\nFinal feature set ({len(available_prep)} features):")
print(f"  Behavioral:    {[c for c in TILT_BEHAVIORAL if c in available_prep]}")
print(f"  Context:       {[c for c in TILT_CONTEXT if c in available_prep]}")
print(f"  Physiological: {[c for c in TILT_PHYSIOLOGICAL if c in available_prep]}")
print(f"  Categorical:   ['is_black']")
print(f"  Total rows: {len(df_features)}")

print("\nPreprocessing complete.")

# Algorithm Comparison
*(Note: Code adapted from `3_algorithm_comparison.py`)*

In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

# Use in-memory scaled data from preprocessing step
df_alg = df_features.copy()
print(f"Using {len(df_alg)} rows, {len(df_alg.columns)} features")

# ---------------------------------------------------------------------------
# PCA for visualization
# ---------------------------------------------------------------------------
def run_pca(df: pd.DataFrame, n_components: int = 2) -> np.ndarray:
    pca = PCA(n_components=n_components, random_state=42)
    reduced = pca.fit_transform(df_alg.values)
    explained = pca.explained_variance_ratio_
    print(f"\n  PCA explained variance: {explained[0]:.2%} + {explained[1]:.2%} = {sum(explained):.2%}")
    print(f"  Top components:")
    for i, comp in enumerate(pca.components_[:2]):
        top_idx = np.argsort(np.abs(comp))[-5:][::-1]
        top_features = [(df.columns[j], comp[j]) for j in top_idx]
        print(f"    PC{i+1}: {', '.join(f'{name}({w:+.2f})' for name, w in top_features)}")
    return reduced

def plot_clusters(reduced: np.ndarray, labels: np.ndarray, title: str, filename: str):
    fig, ax = plt.subplots(figsize=(10, 7))
    scatter = ax.scatter(reduced[:, 0], reduced[:, 1],
                         c=labels, cmap="tab10", alpha=0.3, s=5)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(title)
    fig.colorbar(scatter, ax=ax, label="Cluster/Label")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / filename, dpi=150)
    print(f"  Saved: {PLOT_DIR / filename}")
    plt.close(fig)
    
reduced_data = run_pca(df_alg)

In [43]:
# ---------------------------------------------------------------------------
# K-Means
# ---------------------------------------------------------------------------
print("\n=== K-Means Clustering ===")

# Elbow method
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(df_alg.values)
    inertias.append(km.inertia_)
    sil = silhouette_score(df_alg.values, labels, sample_size=min(10000, len(df_alg)))
    silhouettes.append(sil)
    print(f"  k={k}: inertia={km.inertia_:,.0f}, silhouette={sil:.4f}")

# Plot elbow
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(list(K_range), inertias, "bo-")
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")
ax1.set_title("Elbow Method")
ax2.plot(list(K_range), silhouettes, "ro-")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score vs k")
fig.tight_layout()
fig.savefig(PLOT_DIR / "kmeans_elbow.png", dpi=150)
print(f"  Saved: {PLOT_DIR / 'kmeans_elbow.png'}")
plt.close(fig)

# Best k by silhouette
best_k = list(K_range)[np.argmax(silhouettes)]
print(f"  Best k by silhouette: {best_k}")

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_labels = km_best.fit_predict(df_alg.values)

plot_clusters(reduced_data, km_labels, f"K-Means (k={best_k})", "kmeans_pca.png")

# Cluster profiles
df_temp = df_alg.copy()
df_temp["cluster"] = km_labels
print(f"\n  Cluster sizes: {dict(pd.Series(km_labels).value_counts().sort_index())}")
print("\n  Cluster means (key tilt features):")
tilt_cols = [c for c in ["consecutive_losses_pregame", "avg_tpm_seconds_player",
                          "blunder_cnt_player", "acpl_player", "accuracy_player"]
             if c in df_alg.columns]
if tilt_cols:
    print(df_temp.groupby("cluster")[tilt_cols].mean().round(3).to_string())


=== K-Means Clustering ===
  k=2: inertia=485,466, silhouette=0.1679
  k=3: inertia=453,435, silhouette=0.1036
  k=4: inertia=436,568, silhouette=0.0867
  k=5: inertia=413,798, silhouette=0.0885
  k=6: inertia=408,081, silhouette=0.0903
  k=7: inertia=386,217, silhouette=0.0931
  k=8: inertia=374,384, silhouette=0.0804
  Saved: c:\swe_repos\my-portfolio-and-vps-repos\Chess-Assistance\machine_learning\trainers\trainer-angriness-predictor\notebooks\plots\kmeans_elbow.png
  Best k by silhouette: 2
  Saved: c:\swe_repos\my-portfolio-and-vps-repos\Chess-Assistance\machine_learning\trainers\trainer-angriness-predictor\notebooks\plots\kmeans_pca.png

  Cluster sizes: {0: np.int64(5741), 1: np.int64(19508)}

  Cluster means (key tilt features):
         consecutive_losses_pregame  avg_tpm_seconds_player  blunder_cnt_player  acpl_player  accuracy_player
cluster                                                                                                      
0                             0.

In [44]:
# ---------------------------------------------------------------------------
# Isolation Forest
# ---------------------------------------------------------------------------
print("\n=== Isolation Forest (Anomaly Detection) ===")

results = {}
for contamination in [0.05, 0.10, 0.15, 0.20]:
    iso = IsolationForest(
        contamination=contamination,
        random_state=42,
        n_estimators=200,
    )
    labels = iso.fit_predict(df_alg.values)  # 1=normal, -1=anomaly
    n_anomalies = (labels == -1).sum()
    scores = iso.decision_function(df_alg.values)
    results[contamination] = {
        "labels": labels,
        "scores": scores,
        "n_anomalies": n_anomalies,
    }
    print(f"  contamination={contamination:.0%}: {n_anomalies} anomalies ({n_anomalies/len(df_alg):.1%})")

# Use 10% as default visualization
default = results[0.10]
is_anomaly = (default["labels"] == -1).astype(int)
plot_clusters(reduced_data, is_anomaly, "Isolation Forest (10% contamination)", "iforest_pca.png")

# Anomaly profile vs normal
df_temp = df_alg.copy()
df_temp["is_anomaly"] = is_anomaly
tilt_cols = [c for c in ["consecutive_losses_pregame", "avg_tpm_seconds_player",
                          "blunder_cnt_player", "acpl_player", "accuracy_player",
                          "avg_ppm", "avg_celsius", "water_intake_ml"]
             if c in df_alg.columns]
if tilt_cols:
    print("\n  Normal vs Anomaly means (key features):")
    print(df_temp.groupby("is_anomaly")[tilt_cols].mean().round(3).to_string())

# Score distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(default["scores"], bins=80, edgecolor="white", alpha=0.8)
ax.axvline(x=0, color="red", linestyle="--", label="threshold")
ax.set_xlabel("Anomaly Score")
ax.set_ylabel("Count")
ax.set_title("Isolation Forest Score Distribution")
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "iforest_scores.png", dpi=150)
print(f"  Saved: {PLOT_DIR / 'iforest_scores.png'}")
plt.close(fig)


=== Isolation Forest (Anomaly Detection) ===
  contamination=5%: 1263 anomalies (5.0%)
  contamination=10%: 2525 anomalies (10.0%)
  contamination=15%: 3788 anomalies (15.0%)
  contamination=20%: 5050 anomalies (20.0%)
  Saved: c:\swe_repos\my-portfolio-and-vps-repos\Chess-Assistance\machine_learning\trainers\trainer-angriness-predictor\notebooks\plots\iforest_pca.png

  Normal vs Anomaly means (key features):
            consecutive_losses_pregame  avg_tpm_seconds_player  blunder_cnt_player  acpl_player  accuracy_player  avg_ppm  avg_celsius  water_intake_ml
is_anomaly                                                                                                                                             
0                               -0.080                   0.043              -0.070       -0.043           -0.073   -0.028       -0.001           -0.014
1                                0.716                  -0.386               0.627        0.390            0.657    0.249        

In [45]:
# ---------------------------------------------------------------------------
# DBSCAN
# ---------------------------------------------------------------------------
print("\n=== DBSCAN ===")

# DBSCAN on PCA-reduced data (full-dimensional DBSCAN is unreliable)
for eps in [1.0, 1.5, 2.0, 2.5]:
    db = DBSCAN(eps=eps, min_samples=10)
    labels = db.fit_predict(reduced_data)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f"  eps={eps}: {n_clusters} clusters, {n_noise} noise points ({n_noise/len(df_alg):.1%})")

    if n_clusters >= 2:
        mask = labels != -1
        if mask.sum() > 1:
            sil = silhouette_score(reduced_data[mask], labels[mask],
                                    sample_size=min(10000, mask.sum()))
            print(f"    silhouette (excl. noise): {sil:.4f}")

# Visualize eps=2.0 as a reasonable middle ground
db = DBSCAN(eps=2.0, min_samples=10)
db_labels = db.fit_predict(reduced_data)
plot_clusters(reduced_data, db_labels, "DBSCAN (eps=2.0)", "dbscan_pca.png")

print("\n" + "=" * 60)
print("SUMMARY — Pick the approach that best separates tilt patterns:")
print("  - K-Means: Good if clusters show clear behavioral differences")
print("    (high ACPL/blunders in one cluster, low in another)")
print("  - Isolation Forest: Good if tilt is rare and anomalous")
print("    (check if anomalies have high consecutive_losses + high ACPL)")
print("  - DBSCAN: Good if tilt forms dense pockets in feature space")
print("=" * 60)


=== DBSCAN ===
  eps=1.0: 1 clusters, 46 noise points (0.2%)
  eps=1.5: 2 clusters, 23 noise points (0.1%)
    silhouette (excl. noise): 0.6637
  eps=2.0: 1 clusters, 18 noise points (0.1%)
  eps=2.5: 2 clusters, 4 noise points (0.0%)
    silhouette (excl. noise): 0.8260
  Saved: c:\swe_repos\my-portfolio-and-vps-repos\Chess-Assistance\machine_learning\trainers\trainer-angriness-predictor\notebooks\plots\dbscan_pca.png

SUMMARY — Pick the approach that best separates tilt patterns:
  - K-Means: Good if clusters show clear behavioral differences
    (high ACPL/blunders in one cluster, low in another)
  - Isolation Forest: Good if tilt is rare and anomalous
    (check if anomalies have high consecutive_losses + high ACPL)
  - DBSCAN: Good if tilt forms dense pockets in feature space


# Tune and Select
*(Note: Code adapted from `4_tune_and_select.py`)*

In [ ]:
import json

TILT_FEATURES = [
    "consecutive_losses_pregame",
    "avg_tpm_seconds_player",
    "blunder_cnt_player",
    "acpl_player",
    "accuracy_player",
]

# Use in-memory data from preprocessing step
scaled = df_features.copy()
unscaled = df_unscaled.copy()

print(f"Using {len(scaled)} rows, {len(scaled.columns)} features")

In [ ]:
# ---------------------------------------------------------------------------
# K-Means tuning
# ---------------------------------------------------------------------------
print("\n=== K-Means Hyperparameter Tuning ===")

results_km = []
for k in range(2, 8):
    for init_method in ["k-means++", "random"]:
        for n_init in [10, 20]:
            km = KMeans(
                n_clusters=k,
                init=init_method,
                n_init=n_init,
                max_iter=500,
                random_state=42,
            )
            labels = km.fit_predict(scaled.values)
            sil = silhouette_score(
                scaled.values, labels,
                sample_size=min(10_000, len(scaled)),
            )
            results_km.append({
                "k": k,
                "init": init_method,
                "n_init": n_init,
                "silhouette": round(sil, 5),
                "inertia": round(km.inertia_, 1),
            })
            print(f"  k={k}, init={init_method}, n_init={n_init} → "
                  f"sil={sil:.4f}, inertia={km.inertia_:,.0f}")

results_km_df = pd.DataFrame(results_km).sort_values("silhouette", ascending=False)
km_best = results_km_df.iloc[0]
print(f"\n  Best K-Means config:")
print(f"    k={int(km_best['k'])}, init={km_best['init']}, n_init={int(km_best['n_init'])}")
print(f"    silhouette={km_best['silhouette']:.5f}")

print("\n  Top 5 configs:")
print(results_km_df.head(5).to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Isolation Forest tuning
# ---------------------------------------------------------------------------
print("\n=== Isolation Forest Hyperparameter Tuning ===")

tilt_cols = [c for c in TILT_FEATURES if c in scaled.columns]
results_iso = []

for contamination in [0.03, 0.05, 0.08, 0.10, 0.15, 0.20]:
    for n_estimators in [100, 200, 300]:
        for max_features in [0.5, 0.75, 1.0]:
            iso = IsolationForest(
                contamination=contamination,
                n_estimators=n_estimators,
                max_features=max_features,
                random_state=42,
            )
            labels = iso.fit_predict(scaled.values)
            scores = iso.decision_function(scaled.values)

            n_anomalies = (labels == -1).sum()
            anomaly_ratio = n_anomalies / len(scaled)

            score_sep = 0.0
            if tilt_cols and unscaled is not None:
                normal_mask = labels == 1
                anomaly_mask = labels == -1
                if anomaly_mask.sum() > 0:
                    normal_means = unscaled.loc[normal_mask, tilt_cols].mean()
                    anomaly_means = unscaled.loc[anomaly_mask, tilt_cols].mean()
                    diffs = {}
                    for col in tilt_cols:
                        if col == "accuracy_player":
                            diffs[col] = normal_means[col] - anomaly_means[col]
                        else:
                            diffs[col] = anomaly_means[col] - normal_means[col]
                    score_sep = np.mean(list(diffs.values()))

            results_iso.append({
                "contamination": contamination,
                "n_estimators": n_estimators,
                "max_features": max_features,
                "n_anomalies": n_anomalies,
                "anomaly_ratio": round(anomaly_ratio, 4),
                "tilt_separation": round(score_sep, 4),
                "mean_anomaly_score": round(float(scores[labels == -1].mean()), 4) if n_anomalies > 0 else 0,
                "mean_normal_score": round(float(scores[labels == 1].mean()), 4),
            })

results_iso_df = pd.DataFrame(results_iso).sort_values("tilt_separation", ascending=False)

print("\n  Top 5 configs by tilt feature separation:")
print(results_iso_df.head(5).to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Final model profile
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("FINAL PROFILE — Best configurations")
print("=" * 60)

best_k = int(km_best["k"])
km = KMeans(
    n_clusters=best_k,
    init=km_best["init"],
    n_init=int(km_best["n_init"]),
    max_iter=500,
    random_state=42,
)
km_labels_final = km.fit_predict(scaled.values)

print(f"\n--- K-Means (k={best_k}) ---")
print(f"  Cluster sizes: {dict(pd.Series(km_labels_final).value_counts().sort_index())}")

if unscaled is not None:
    tilt_cols = [c for c in TILT_FEATURES if c in unscaled.columns]
    if tilt_cols:
        profile = unscaled.copy()
        profile["cluster"] = km_labels_final
        print("\n  Cluster means (unscaled):")
        print(profile.groupby("cluster")[tilt_cols].mean().round(2).to_string())

iso_best = results_iso_df.iloc[0]
iso = IsolationForest(
    contamination=iso_best["contamination"],
    n_estimators=int(iso_best["n_estimators"]),
    max_features=iso_best["max_features"],
    random_state=42,
)
iso_labels_final = iso.fit_predict(scaled.values)

print(f"\n--- Isolation Forest (contamination={iso_best['contamination']}) ---")
print(f"  Anomalies: {(iso_labels_final == -1).sum()} / {len(scaled)}")

if unscaled is not None:
    tilt_cols = [c for c in TILT_FEATURES if c in unscaled.columns]
    if tilt_cols:
        profile = unscaled.copy()
        profile["is_anomaly"] = (iso_labels_final == -1).astype(int)
        print("\n  Normal vs Anomaly means (unscaled):")
        print(profile.groupby("is_anomaly")[tilt_cols].mean().round(2).to_string())

# PCA visualization of both
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
reduced = pca.fit_transform(scaled.values)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

scatter1 = ax1.scatter(reduced[:, 0], reduced[:, 1],
                       c=km_labels_final, cmap="tab10", alpha=0.3, s=5)
ax1.set_xlabel("PC1")
ax1.set_ylabel("PC2")
ax1.set_title(f"K-Means (k={best_k})")
fig.colorbar(scatter1, ax=ax1, label="Cluster")

iso_colors = (iso_labels_final == -1).astype(int)
scatter2 = ax2.scatter(reduced[:, 0], reduced[:, 1],
                       c=iso_colors, cmap="RdYlGn_r", alpha=0.3, s=5)
ax2.set_xlabel("PC1")
ax2.set_ylabel("PC2")
ax2.set_title(f"Isolation Forest (c={iso_best['contamination']})")
fig.colorbar(scatter2, ax=ax2, label="Anomaly")

fig.suptitle("Best Configurations — Side by Side", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(PLOT_DIR / "best_models_pca.png", dpi=150)
print(f"\n  Saved: {PLOT_DIR / 'best_models_pca.png'}")
plt.close(fig)

# Display config summary
config = {
    "kmeans": {
        "k": best_k,
        "init": km_best["init"],
        "n_init": int(km_best["n_init"]),
        "silhouette": float(km_best["silhouette"]),
    },
    "isolation_forest": {
        "contamination": float(iso_best["contamination"]),
        "n_estimators": int(iso_best["n_estimators"]),
        "max_features": float(iso_best["max_features"]),
        "tilt_separation": float(iso_best["tilt_separation"]),
    },
}
print(f"\n  Best config:\n{json.dumps(config, indent=2)}")

print("\n" + "=" * 60)
print("NEXT STEPS:")
print("  1. Review cluster profiles — does one cluster show clear tilt?")
print("  2. Review anomaly profiles — do anomalies have high ACPL/blunders?")
print("  3. Pick the approach and build production steps/ pipeline")
print("  4. Consider: K-Means labels → supervised classifier (RandomForest)")
print("     or Isolation Forest anomaly score → angriness scale (1-5)")
print("=" * 60)